# Proyecto final · Titanic

**Estadística Descriptiva e Inferencial**

Integrantes: _______________________________     Fecha: ____________

---

## Qué se te pide

Construir un modelo de clasificación **paso a paso**, midiendo qué aporta cada variable
que agregas, y presentar tus hallazgos en 5 slides.

> **No se evalúa qué tan bueno sea tu modelo.** Se evalúa qué entendiste: por qué cada
> variable aporta o no, qué pasa cuando falta una, y cómo justificas tus decisiones.

## Las tres partes

| Parte | Min | Qué haces |
|---|---|---|
| 1 | 45 | Reconocimiento: mirar los datos y escribir tres hipótesis |
| 2 | 100 | **Seis modelos incrementales y qué cambia en cada paso** |
| 3 | 45 | Elegir un umbral y justificarlo |

Después, la presentación: 5 slides, 7 minutos.

## Cómo usar este notebook

Las celdas marcadas con `# ── TU CÓDIGO ──` son las que tienes que completar. Las celdas
de texto con `**Escribe aquí:**` esperan tu respuesta escrita — **esas también se
califican**, y varias valen más que el código.

---
## Celda 0 · Preparación

Ejecuta esta celda tal cual. Carga los datos y prepara las variables que vas a usar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, roc_auc_score)
import warnings
warnings.filterwarnings("ignore")

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

URL = ("https://raw.githubusercontent.com/josefrodrim/"
       "Estad-stica-Descriptiva-E-Inferencial/main/"
       "Proyecto_final_titanic/Data/Titanic-Dataset.csv")

try:
    titanic = pd.read_csv(URL)
    print("Datos cargados desde el repo del curso.")
except Exception:
    titanic = pd.read_csv("Titanic-Dataset.csv")
    print("Datos cargados desde archivo local.")

# ── Variables ya preparadas para ti ──────────────────────────────────────
d = titanic.copy()
d["mujer"] = (d["Sex"] == "female").astype(int)
d["edad"] = d["Age"].fillna(d["Age"].median())     # <- ojo con esto, ver Parte 1
d["familia"] = d["SibSp"] + d["Parch"]             # hermanos/pareja + padres/hijos
d["tiene_camarote"] = d["Cabin"].notna().astype(int)

print(f"\n{len(d)} pasajeros")
print(d[["Survived", "mujer", "Pclass", "edad", "familia", "tiene_camarote"]].head())

### Las variables que vas a usar

| Variable | Qué es |
|---|---|
| `Survived` | **1 sobrevivió, 0 no.** Es lo que predices |
| `mujer` | 1 si es mujer |
| `Pclass` | clase del billete: 1, 2 o 3 |
| `edad` | edad, con los faltantes rellenados con la mediana |
| `familia` | `SibSp + Parch`: cuántos familiares viajaban con esa persona |
| `tiene_camarote` | 1 si la columna `Cabin` no está vacía |

---
# Parte 1 · Reconocimiento  ·  45 min  ·  15 puntos

Antes de modelar, mira los datos.

### 1.1 — Valores faltantes

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Cuenta los valores faltantes de cada columna del dataset ORIGINAL (titanic).
# Muestra solo las que tienen alguno.


### 1.2 — Tasas de supervivencia por grupo

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Calcula la tasa de supervivencia:
#   a) general
#   b) por sexo
#   c) por clase
#   d) por tamaño de familia


### 1.3 — Dos gráficos

Haz al menos dos. Sugerencias: tasa de supervivencia por clase y sexo, distribución de
edades por resultado, tasa por tamaño de familia.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────


### 1.4 — Tus tres hipótesis  ·  **se califica**

Escríbelas **ahora**, antes de modelar nada. Una frase cada una.

**Escribe aquí:**

> **Hipótesis 1:**
>
> **Hipótesis 2:**
>
> **Hipótesis 3:**

*(Al final del proyecto vas a volver aquí para ver cuáles se cumplieron.)*

---
# Parte 2 · Seis modelos incrementales  ·  100 min  ·  40 puntos

**Esta es la parte central del proyecto.**

Vas a construir seis modelos, agregando una variable cada vez, y ver qué cambia.

### 2.1 — Separa los datos

Usa `test_size=0.3`, `random_state=42` y `stratify=y`. Es importante que uses estos
valores para que tu trabajo sea comparable con el de las otras parejas.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
X = d[["mujer", "Pclass", "edad", "familia", "tiene_camarote"]]
y = d["Survived"]

X_train, X_test, y_train, y_test = None, None, None, None

print(f"train={len(X_train)}  test={len(X_test)}")

### 2.2 — La tabla de los seis modelos

| Modelo | Variables |
|---|---|
| M0 | ninguna: predecir siempre la clase mayoritaria |
| M1 | `mujer` |
| M2 | `mujer`, `Pclass` |
| M3 | `mujer`, `Pclass`, `edad` |
| M4 | `mujer`, `Pclass`, `edad`, `familia` |
| M5 | `mujer`, `Pclass`, `edad`, `familia`, `tiene_camarote` |

Para cada uno registra el **AUC** y la **exactitud**, los dos **en prueba**.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
modelos = {
    "M1": ["mujer"],
    "M2": ["mujer", "Pclass"],
    "M3": ["mujer", "Pclass", "edad"],
    "M4": ["mujer", "Pclass", "edad", "familia"],
    "M5": ["mujer", "Pclass", "edad", "familia", "tiene_camarote"],
}

filas = []
# M0: la regla trivial. ¿Qué exactitud tiene predecir siempre "murió"?
filas.append({"modelo": "M0", "variables": 0, "AUC": None, "exactitud": None})

for nombre, cols in modelos.items():
    # entrena, predice sobre PRUEBA, y guarda AUC y exactitud
    pass

tabla = pd.DataFrame(filas)
print(tabla)

### 2.3 — Las tres preguntas  ·  **se califican**

**a) ¿Qué variable aportó más? ¿Y cuál casi nada?**

**Escribe aquí:**

>

**b) ¿Alguna hizo empeorar el modelo? ¿Cómo es posible que agregar información empeore?**

**Escribe aquí:**

>


### 2.4 — El hallazgo  ·  **20 puntos**

Ahora compara **los coeficientes de una misma variable entre modelos distintos**.

Busca una variable cuyo coeficiente cambie de forma llamativa —de tamaño, o incluso de
signo— al agregar otra variable al modelo.

**Pista:** fíjate en las variables que parecían no importar cuando estaban solas.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Ajusta modelos con distintas combinaciones y compara el coeficiente
# de UNA MISMA variable entre ellos.
#
# Sugerencia: prueba con 'familia' sola, y después acompañada.


**c) Describe el hallazgo: qué variable, cómo cambió, y por qué crees que pasa.**

**Escribe aquí:**

>
>
>


---
# Parte 3 · El umbral  ·  45 min  ·  15 puntos

## El escenario

Una **aseguradora** debe estimar cuántos pasajeros sobrevivirán, para saber cuánto dinero
provisionar para indemnizaciones a las familias de los fallecidos.

| Error | Qué pasa |
|---|---|
| **Falso positivo** — predije que sobrevive y murió | No se provisionó. Hay que pagar sin reserva: problema financiero serio |
| **Falso negativo** — predije que muere y sobrevivió | Se provisionó de más. Dinero inmovilizado que no se pudo usar |

Los dos cuestan, y no lo mismo.

Usa el mejor modelo de la Parte 2 (por AUC).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
# Prueba al menos cinco umbrales. Para cada uno muestra:
# VP, FP, FN, VN, exactitud, precisión y recall.


### 3.1 — Tu decisión  ·  **se califica**

**¿Qué umbral eliges y por qué? Conecta tu respuesta con el escenario de la aseguradora,
no con la exactitud.**

**Escribe aquí:**

> **Umbral elegido:**
>
> **Justificación:**
>
>


---
# Cierre  ·  15 puntos

### Vuelve a tus hipótesis

Regresa a la sección 1.4 y compara. **¿Cuáles se cumplieron y cuáles no?**

**Escribe aquí:**

>


### Tres limitaciones de tu análisis  ·  **15 puntos**

Piensa en qué NO permite concluir este trabajo. Algunas pistas por si te ayudan: los
valores faltantes de edad, el tamaño de la muestra de prueba, la diferencia entre
correlación y causalidad, y qué pasaría si aplicaras este modelo a otro barco.

**Escribe aquí:**

> **Limitación 1:**
>
> **Limitación 2:**
>
> **Limitación 3:**


---

## Antes de entregar

- [ ] Las tres hipótesis están escritas **antes** de la sección de modelos
- [ ] Todas mis métricas son de **prueba**, y lo digo explícitamente
- [ ] Comparé mi exactitud contra la regla trivial de M0
- [ ] La tabla de los seis modelos está completa
- [ ] Identifiqué un coeficiente que cambia y expliqué por qué
- [ ] Justifiqué el umbral con el escenario, no con la exactitud
- [ ] Escribí tres limitaciones reales
- [ ] La presentación tiene exactamente 5 slides

## Y la presentación

| # | Slide |
|---|---|
| 1 | El problema y los datos |
| 2 | Lo que vimos antes de modelar (tus hipótesis) |
| 3 | La tabla de los seis modelos |
| 4 | **El hallazgo** |
| 5 | La decisión y sus límites |

7 minutos. La slide 4 es la que más pesa.

---
*Estadística Descriptiva e Inferencial · Proyecto final*